# Network Expansion
# Model 1 - Deterministic Baseline

In [33]:
import os 
os.chdir("..")

In [34]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import Model, GRB, quicksum

from src.classes import DistributionNetwork
from src.classes import Substation

![Topology example](../figures/Model1_Initial.png)  
*Example topology of a distribution system evaluated in this notebook.*

Defining the Distribution Network: Input data

In [35]:
NODES = [f"N{i}" for i in range(1,14)] # Listing system nodes: 'N1','N2', ..., 'N13'
LOADS = [f'D{i}' for i in range(1,11)] # Listing system loads: 'D1', 'D2', ..., 'D10'
# NOTE: Candidate nodes (ones with potential substations) are added separately using add_candidate_substations method.

# Initial substations
S1 = Substation("S1", "N4", 40, ["N3", "N5", "N9"]) # Substation S1 at node N4, with capacity 40, connected to nodes N3, N5, N9
SUBSTATIONS = [S1] # Substations to pass to a class

load_capacity = {'D1': 5,
                 'D2': 2,
                 'D3': 2,
                 'D4': 5,
                 'D5': 3,
                 'D6': 2,
                 'D7': 6,
                 'D8': 5,
                 'D9': 3,
                 'D10': 4}

Mapping (nodal locations)

In [36]:
# Loads
loads_locations = {
    "D1": "N1",
    "D2": "N2",
    "D3": "N3",
    "D4": "N6",
    "D5": "N7",
    "D6": "N8",
    "D7": "N9",
    "D8": "N11",
    "D9": "N12",
    "D10": "N13",
}

# Distribution lines
nodes_connected = {
    "N1": ["N2"],
    "N2": ["N1", "N3"],
    "N3": ["N2", "N4"],
    "N4": ["N3", "N5", "N9"],
    "N5": ["N4", "N6"],
    "N6": ["N5","N7", "N8"],
    "N7": ["N6"],
    "N8": ["N6"],
    "N9": ["N4", "N10"],
    "N10": ["N9", "N11", "N13"],
    "N11": ["N10", "N12"],
    "N12": ["N11"],
    "N13": ["N10"]
}

Instancing a pre-defined Distribution Network

In [37]:
DistributionNetwork = DistributionNetwork(NODES, 
                                          LOADS, 
                                          SUBSTATIONS, 
                                          load_capacity,
                                          nodes_connected,
                                          loads_locations)

In [38]:
# TODO: (optional) Draw current Distribution Network

Adding *candidate* substations for expansion (with potential connection lines)

In [39]:
capacity = 20
S2 = Substation("S2", "N14", capacity, ["N2"], fix_cost = 100, edge_cost = 1.0) # Potential substation S2 at node N14, with potential connection to node N2
S3 = Substation("S3", "N15", capacity, ["N5"], fix_cost = 100, edge_cost = 1.0)
S4 = Substation("S4", "N16", capacity, ["N11", "N13"], fix_cost = 100, edge_cost = 1.0)
DistributionNetwork.add_candidate_substations([S2, S3, S4])

Defining a Network Expansion optimization problem (also in `solver.py`)

1. Index sets
2. Model. Parameters
3. Model. Decision variables
4. Model. Constraints
5. Model. Objective function

In [40]:
def solve_network(Network: DistributionNetwork, OutputFlag = 0):
    # ---------------
    #  Index sets
    # ---------------
    N = list(np.arange(1, len(Network.NODES)+1))        # List of node indices
    S = list(np.arange(1, len(Network.SUBSTATIONS)+1))  # List of substations indices

    # Demand vector (demand of each node)
    d = np.zeros(len(Network.NODES))
    for load, node in Network.loads_locations.items():
        ind = int(node[1:]) - 1
        d[ind] = Network.load_capacity[load]

    # Substations nodes
    S_all_nodes = [int(sub.node[1:]) for sub in Network.SUBSTATIONS]  # all substations nodes

    # Demand/non-substation nodes
    D = np.delete(np.arange(1, len(Network.NODES)+1), np.array(S_all_nodes)-1)  # only nodes without substations

    # ----------------------
    #   Model
    # ----------------------
    model = gp.Model("Radial_Distribution_Network")

    #   Model. Parameters
    M = float(np.sum(d)) # Big-M for flow

    #   Model. Decision variables
    w = model.addVars(S, vtype=GRB.BINARY, name="w")
    y = model.addVars([(i,s) for i in N for s in S], vtype=GRB.BINARY, name="y")
    x = model.addVars([(i,j,s) for (i,j) in Network.A for s in S], vtype=GRB.BINARY, name="x")
    f = model.addVars([(s,i,j) for s in S for (i,j) in Network.A], lb=0.0, ub=M, vtype=GRB.CONTINUOUS, name="f")
    r = model.addVars(S, lb=0.0, vtype=GRB.CONTINUOUS, name="r")

    #   Model. Constraints
    #
    # Power balance
    model.addConstr(quicksum(r[s] for s in S) == np.sum(d), name="power_balance")

    # Node assignment: only demand nodes
    for i in D:
        model.addConstr(quicksum(y[i,s] for s in S) == 1, name=f"assign_{i}")

    for i in N:
        for s in S:
            model.addConstr(y[i,s] <= w[s], name=f"y_le_w_{i}_{s}")

    # Substation node assignment: assigned to self if activated
    for s in S:
        node_idx = int(Network.SUBSTATIONS[s-1].node[1:])
        model.addConstr(y[node_idx,s] == w[s], name=f"substation_assign_{s}")
        # substation nodes cannot be assigned to other substations
        for s2 in S:
            if s2 != s:
                model.addConstr(y[node_idx,s2] == 0, name=f"substation_no_assign_{s}_{s2}")

    # Existing substations must be active (Initial constraint)
    # Looks for substations with ZERO FIXED COST.
    for s in S:
        if Network.SUBSTATIONS[s-1].fix_cost == 0:
            model.addConstr(w[1] == 1, name="w_act_1")

    # Substation supply
    for s in S:
        model.addConstr(r[s] == quicksum(d[i-1]*y[i,s] for i in N), name=f"supply_def_{s}")

    # Capacity
    for s in S:
        model.addConstr(r[s] <= Network.SUBSTATIONS[s-1].capacity * w[s], name=f"capacity_{s}")

    # Flow conservation
    for s in S:
        for i in N:
            incoming = quicksum(f[s,j,i] for (j,k) in Network.A if k==i)
            outgoing = quicksum(f[s,i,j] for (k,j) in Network.A if k==i)
            if i == int(Network.SUBSTATIONS[s-1].node[1:]):  # substation root
                model.addConstr(outgoing - incoming == r[s], name=f"flow_balance_sub_{s}_{i}")
            elif i in D:
                model.addConstr(outgoing - incoming == -d[i-1]*y[i,s], name=f"flow_balance_{s}_{i}")

    # Flow only if arc assigned
    for s in S:
        for (i,j) in Network.A:
            model.addConstr(f[s,i,j] <= M*x[i,j,s], name=f"f_cap_{s}_{i}_{j}")

    # Radiality: each demand node has exactly one parent per assigned substation
    for s in S:
        for i in D:
            model.addConstr(quicksum(x[j,i,s] for (j,k) in Network.A if k==i) == y[i,s], name=f"one_parent_{s}_{i}")
        # substation node has no parent
        node_idx = int(Network.SUBSTATIONS[s-1].node[1:])
        model.addConstr(quicksum(x[j,node_idx,s] for (j,k) in Network.A if k==node_idx) == 0, name=f"parent_root_{s}")

    # Tree size: arcs = nodes assigned - w[s]
    for s in S:
        model.addConstr(quicksum(x[i,j,s] for (i,j) in Network.A) == quicksum(y[k,s] for k in N) - w[s], name=f"tree_size_{s}")

    #   Model. Objective function
    # 
    #   f(x) = fixed cost + edge cost
    fix_term = quicksum(Network.SUBSTATIONS[s-1].fix_cost * w[s] for s in S)
    edge_term = 0.5*quicksum(Network.edge_cost[(min(i,j),max(i,j))]*x[i,j,s] for (i,j) in Network.A for s in S)
    model.setObjective(fix_term + edge_term, GRB.MINIMIZE)

    # ------------------------
    #   Solve
    # ------------------------
    model.setParam('OutputFlag', OutputFlag)
    model.optimize()

    # Extract numerical results
    w_val = {s: w[s].X for s in S}
    y_val = {(i, s): y[i, s].X for i in N for s in S}
    x_val = {(i, j, s): x[i, j, s].X for (i, j) in Network.A for s in S}
    f_val = {(s, i, j): f[s, i, j].X for s in S for (i, j) in Network.A}
    r_val = {s: r[s].X for s in S}

    # Return numerical data
    return {
        "objective": model.ObjVal,
        "w": w_val,
        "y": y_val,
        "x": x_val,
        "f": f_val,
        "r": r_val
    }

In [41]:
solution = solve_network(DistributionNetwork, OutputFlag=1)

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-9300H CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 338 rows, 328 columns and 1118 nonzeros
Model fingerprint: 0x35e26842
Variable types: 132 continuous, 196 integer (196 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [5e-01, 1e+02]
  Bounds range     [1e+00, 4e+01]
  RHS range        [1e+00, 4e+01]
Presolve removed 283 rows and 275 columns
Presolve time: 0.00s
Presolved: 55 rows, 53 columns, 179 nonzeros
Variable types: 20 continuous, 33 integer (31 binary)
Found heuristic solution: objective 100.0000000

Root relaxation: objective 0.000000e+00, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Inc

Printing results

In [42]:
from src.solver import print_results

print_results(DistributionNetwork, solution)


Substation activation and supply:
S1 at N4: w=1.0, r=37.0
S2 at N14: w=-0.0, r=0.0
S3 at N15: w=0.0, r=0.0
S4 at N16: w=-0.0, r=0.0

Node assignments:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1

Arcs used:
Arc N2 -> N1 assigned to S1
Arc N3 -> N2 assigned to S1
Arc N4 -> N3 assigned to S1
Arc N4 -> N5 assigned to S1
Arc N4 -> N9 assigned to S1
Arc N5 -> N6 assigned to S1
Arc N6 -> N7 assigned to S1
Arc N6 -> N8 assigned to S1
Arc N9 -> N10 assigned to S1
Arc N10 -> N11 assigned to S1
Arc N10 -> N13 assigned to S1
Arc N11 -> N12 assigned to S1


Solving the optimization problem for next years.  
Using previous years as new Distribution Network input (deterministic model).  
**Increasing demand** in each demand node uniformly **by 5%**.

In [43]:
# Increasing demand
demand_rate = 0.05 # = 5%
for load, demand in DistributionNetwork.load_capacity.items():
    DistributionNetwork.load_capacity[load] = (1 + demand_rate) * demand

In [44]:
# Solving the problem
solution_2 = solve_network(DistributionNetwork)

# Updating initial substation activation constraints for the next time period
DistributionNetwork.update_substations(solution_2['w']) 

In [45]:
print("Year 2")
print_results(DistributionNetwork, solution_2)

Year 2

Substation activation and supply:
S1 at N4: w=1.0, r=38.85
S2 at N14: w=-0.0, r=0.0
S3 at N15: w=0.0, r=0.0
S4 at N16: w=-0.0, r=0.0

Node assignments:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1

Arcs used:
Arc N2 -> N1 assigned to S1
Arc N3 -> N2 assigned to S1
Arc N4 -> N3 assigned to S1
Arc N4 -> N5 assigned to S1
Arc N4 -> N9 assigned to S1
Arc N5 -> N6 assigned to S1
Arc N6 -> N7 assigned to S1
Arc N6 -> N8 assigned to S1
Arc N9 -> N10 assigned to S1
Arc N10 -> N11 assigned to S1
Arc N10 -> N13 assigned to S1
Arc N11 -> N12 assigned to S1


In [46]:
# Years 3-10
S = list(np.arange(1, len(DistributionNetwork.SUBSTATIONS)+1))
N = list(np.arange(1, len(DistributionNetwork.NODES)+1))  # [1,2, ...]

for i in range(3,11,1):
    for load, demand in DistributionNetwork.load_capacity.items():
        DistributionNetwork.load_capacity[load] = (1 + demand_rate) * demand    # Increase demand
    sol = solve_network(DistributionNetwork)                                    # Solve
    print(f"\nYear {i}")
    print("Substation activation and supply:")
    for s in S:
        print(f"{DistributionNetwork.SUBSTATIONS[s-1].id} at {DistributionNetwork.SUBSTATIONS[s-1].node}: w={sol['w'][s]}, r={sol['r'][s]}")
    DistributionNetwork.update_substations(sol['w'])                            # Update conditions
        


Year 3
Substation activation and supply:
S1 at N4: w=1.0, r=33.075
S2 at N14: w=1.0, r=7.7175
S3 at N15: w=0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Year 4
Substation activation and supply:
S1 at N4: w=1.0, r=34.728750000000005
S2 at N14: w=1.0, r=8.103375000000002
S3 at N15: w=-0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Year 5
Substation activation and supply:
S1 at N4: w=1.0, r=36.465187500000006
S2 at N14: w=1.0, r=8.508543750000001
S3 at N15: w=-0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Year 6
Substation activation and supply:
S1 at N4: w=1.0, r=35.73588375000001
S2 at N14: w=1.0, r=11.486534062500002
S3 at N15: w=0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Year 7
Substation activation and supply:
S1 at N4: w=1.0, r=37.522677937500006
S2 at N14: w=1.0, r=12.060860765625002
S3 at N15: w=0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Year 8
Substation activation and supply:
S1 at N4: w=1.0, r=39.39881183437513
S2 at N14: w=1.0, r=12.663903803906253
S3 at N15: w=-0.0, r=0.0
S4 at N16: w=0.0, r=0.0

Year 9
Substation acti

In [47]:
print("\nNode assignments in Year 10:")
for i in N:
    for s in S:
        if sol['y'][i,s] > 0.5:
            print(f"Node {DistributionNetwork.NODES[i-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Node assignments in Year 10:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S3
Node N6 assigned to S3
Node N7 assigned to S3
Node N8 assigned to S3
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1
Node N14 assigned to S2
Node N15 assigned to S3


Results, plots, short insights

In [48]:
# TODO:

![Final example](../figures/Model1_Final.png)  
*Optimization solution.*